# Financial Literacy Chatbot (Africa-Focused)
# Simple chatbot to answer common questions on savings, mobile money, credit, and fraud prevention.


In [1]:
# necessary imports
import os
import random
import json
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import torch
from torch.nn.functional import softmax
import torch.nn as nn
import time
import requests
import re
from deep_translator import GoogleTranslator
import concurrent.futures
from threading import Lock
# Transformers & HF
from transformers import MarianMTModel, MarianTokenizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# Sentence Transformers (for retrieval)
from sentence_transformers import SentenceTransformer, util
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report, confusion_matrix, accuracy_score
from torch.utils.data import DataLoader, TensorDataset

# [ADD RAG ARCHITECTURE IMPORTS]
# Auto-install faiss if not present
import importlib.util
import sys
import subprocess

import faiss
from typing import List, Dict, Tuple


# Add these for better performance monitoring
import warnings
warnings.filterwarnings('ignore')

e:\Fin-Chat\chatbot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Cleaning dataset

In [ ]:
# Define paths
input_file = '../data/Financial_Literacy_FAQs_500.csv'
output_file = '../data/Financial_Literacy_FAQs_Cleaned.csv'

# Read the CSV file
df = pd.read_csv(input_file)

# Strip whitespace from all string columns
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

# Remove duplicate questions (keep first occurrence)
df = df.drop_duplicates(subset=['Question'], keep='first')

# Remove generic placeholder entries (the numbered tips)
df = df[~df['Question'].str.contains(r'What is a common tip to manage money #\d+', regex=True, na=False)]

# Replace empty sources with 'General'
df['Source'] = df['Source'].fillna('General')
df['Source'] = df['Source'].replace('', 'General')

# Fix encoding issues
df['Answer'] = df['Answer'].str.replace('â€"', '-', regex=False)
df['Answer'] = df['Answer'].str.replace('â€™', "'", regex=False)

# Consolidate similar questions with multiple variants into single entries with "Multiple sources"
# Group by Question and if there are multiple sources, combine them
def consolidate_sources(group):
    if len(group) > 1:
        # Multiple entries for same question - mark as Multiple sources
        group.loc[group.index[0], 'Source'] = 'Multiple sources'
        return group.iloc[[0]]
    return group

df = df.groupby('Question', as_index=False).apply(consolidate_sources).reset_index(drop=True)

# Sort by Category and Question for better organization
df = df.sort_values(['Category', 'Question']).reset_index(drop=True)

# Save cleaned CSV
df.to_csv(output_file, index=False)

print(f"Original rows: 500")
print(f"Cleaned rows: {len(df)}")
print(f"\nCleaned file saved as: {output_file}")
print(f"\nBreakdown by category:")
print(df['Category'].value_counts().sort_index())

Original rows: 500
Cleaned rows: 409

Cleaned file saved as: ../data/Financial_Literacy_FAQs_Cleaned.csv

Breakdown by category:
Category
Banking & Mobile Money                   60
Budgeting & Saving                       50
Consumer Protection & Fraud Awareness    45
Credit & Loans                           45
Digital Financial Literacy               50
Financial Planning & Retirement          20
Financial Rights & Regulations           20
Fraud Prevention & Scams                 39
Insurance & Risk                         25
Investing basics                         25
Small Business & Farming Finance         30
Name: count, dtype: int64


# English Dataset Creation

In [ ]:
import pandas as pd
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

# Setup paths
project_root = Path('..')
data_folder = project_root / 'data'
MODELS_DIR = project_root / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("🚀 Creating Financial Literacy FAISS Index with FAQ + Training Guide")

# 1. LOAD CSV FAQ DOCUMENT
print("\n📋 Step 1: Loading CSV FAQ document...")

csv_path = data_folder / "Financial_Literacy_FAQs_Cleaned.csv"
if not csv_path.exists():
    raise FileNotFoundError(f"CSV document not found: {csv_path}")

# Load and process CSV
df = pd.read_csv(csv_path)
print(f"✅ Loaded CSV with {len(df)} Q&A pairs")
print(f"Categories: {df['Category'].unique().tolist()}")

# Convert CSV to Document objects
csv_documents = []
for idx, row in df.iterrows():
    # Create comprehensive text content
    content = f"Category: {row['Category']}\n\nQuestion: {row['Question']}\n\nAnswer: {row['Answer']}"
    
    # Create metadata
    metadata = {
        'source_type': 'faq',
        'document_type': 'FAQ',
        'category': row['Category'],
        'question': row['Question'],
        'source': row['Source'],
        'is_direct_answer': True,
        'row_index': idx
    }
    
    csv_documents.append(Document(page_content=content, metadata=metadata))

print(f"✅ Created {len(csv_documents)} FAQ documents")

# Show sample FAQ
if csv_documents:
    sample = csv_documents[0]
    print(f"\nSample FAQ:")
    print(f"Content: {sample.page_content[:150]}...")
    print(f"Metadata: {sample.metadata}")

# 2. LOAD PDF TRAINING GUIDE

print("\n📚 Step 2: Loading PDF Training Guide...")

pdf_path = data_folder / "FINANCIAL LITERACY.pdf"
if not pdf_path.exists():
    raise FileNotFoundError(f"PDF document not found: {pdf_path}")

# Load the PDF document
loader = PyPDFLoader(str(pdf_path))
pdf_documents = loader.load()

print(f"✅ Loaded PDF document with {len(pdf_documents)} pages")

# Enhance PDF metadata
for doc in pdf_documents:
    doc.metadata['source_type'] = 'training_guide'
    doc.metadata['document_type'] = 'Training_Guide'
    doc.metadata['is_direct_answer'] = False
    
    # Try to identify module from page content
    content_lower = doc.page_content.lower()
    if 'budgeting' in content_lower or 'saving' in content_lower[:500]:
        doc.metadata['module'] = 'Budgeting & Saving'
    elif 'loan' in content_lower[:500] or 'debt' in content_lower[:500]:
        doc.metadata['module'] = 'Loan Management'
    elif 'insurance' in content_lower[:500]:
        doc.metadata['module'] = 'Insurance'
    elif 'investment' in content_lower[:500]:
        doc.metadata['module'] = 'Investment'
    elif 'retirement' in content_lower[:500] or 'old age' in content_lower[:500]:
        doc.metadata['module'] = 'Retirement Planning'
    elif 'bank' in content_lower[:500] or 'mobile money' in content_lower[:500]:
        doc.metadata['module'] = 'Banking & Mobile Money'
    else:
        doc.metadata['module'] = 'General'

print(f"Document info: {pdf_documents[0].metadata}")

# Show a sample of the PDF content
if pdf_documents:
    sample_content = pdf_documents[0].page_content[:200] + "..." if len(pdf_documents[0].page_content) > 200 else pdf_documents[0].page_content
    print(f"Sample content: {sample_content}")

# 3. COMBINE ALL DOCUMENTS
print("\n🔗 Step 3: Combining all documents...")

all_documents = csv_documents + pdf_documents

print(f"✅ Total documents loaded: {len(all_documents)}")
print(f"   - FAQ documents: {len(csv_documents)}")
print(f"   - Training guide pages: {len(pdf_documents)}")

# Summary statistics
print("\n📊 Document Statistics:")
print(f"   Total characters (CSV): {sum(len(doc.page_content) for doc in csv_documents):,}")
print(f"   Total characters (PDF): {sum(len(doc.page_content) for doc in pdf_documents):,}")
print(f"   Average FAQ length: {sum(len(doc.page_content) for doc in csv_documents) // len(csv_documents) if csv_documents else 0} chars")
print(f"   Average PDF page length: {sum(len(doc.page_content) for doc in pdf_documents) // len(pdf_documents) if pdf_documents else 0} chars")

# Show category distribution
print("\n📂 FAQ Category Distribution:")
category_counts = df['Category'].value_counts()
for category, count in category_counts.head(10).items():
    print(f"   - {category}: {count}")

print("\n✅ Document loading complete! Ready for chunking and FAISS creation.")

🚀 Creating Financial Literacy FAISS Index with FAQ + Training Guide

📋 Step 1: Loading CSV FAQ document...
✅ Loaded CSV with 409 Q&A pairs
Categories: ['Banking & Mobile Money', 'Budgeting & Saving', 'Consumer Protection & Fraud Awareness', 'Credit & Loans', 'Digital Financial Literacy', 'Financial Planning & Retirement', 'Financial Rights & Regulations', 'Fraud Prevention & Scams', 'Insurance & Risk', 'Investing basics', 'Small Business & Farming Finance']
✅ Created 409 FAQ documents

Sample FAQ:
Content: Category: Banking & Mobile Money

Question: Can I borrow money using mobile money?

Answer: Yes, some providers allow small loans through mobile walle...
Metadata: {'source_type': 'faq', 'document_type': 'FAQ', 'category': 'Banking & Mobile Money', 'question': 'Can I borrow money using mobile money?', 'source': 'General', 'is_direct_answer': True, 'row_index': 0}

📚 Step 2: Loading PDF Training Guide...
✅ Loaded PDF document with 302 pages
Document info: {'producer': 'PDFium', 'creat

# Chichewa Translation

In [ ]:

# IMPROVED CHICHEWA TRANSLATION WITH QUALITY CONTROL

print("🚀 Starting IMPROVED Chichewa Translation with Quality Control...")

# Load English dataset
en_df = pd.read_csv(data_folder / "financial_faqs_en_500.csv")
print(f"✅ Loaded {len(en_df)} English FAQs")

# Initialize translators
en_to_ny = GoogleTranslator(source='en', target='ny')
ny_to_en = GoogleTranslator(source='ny', target='en')  # For back-translation

# EXPANDED FINANCIAL GLOSSARY (Critical for accuracy)
financial_glossary = {
    # Banking & Accounts
    "bank": "banki",
    "account": "akaunti",
    "savings account": "akaunti yosunga ndalama",
    "bank account": "akaunti ya ku banki",
    
    # Money & Finance
    "money": "ndalama",
    "cash": "ndalama",
    "income": "ndalama zomwe mumalandira",
    "expenses": "ndalama zomwe mumagwiritsa ntchito",
    "salary": "malipiro",
    "payment": "malipiro",
    
    # Saving & Budgeting
    "budget": "bajeti",
    "save": "sunga",
    "saving": "kusunga ndalama",
    "savings": "ndalama zosungidwa",
    "emergency fund": "ndalama za mwadzidzidzi",
    
    # Loans & Credit
    "loan": "ngongole",
    "borrow": "kubwereketsa",
    "debt": "ngongole",
    "credit": "ngongole",
    "interest": "chiwongoladzanja",
    "interest rate": "mtengo wa chiwongoladzanja",
    "collateral": "chikole",
    "apply for loan": "kupempha ngongole",
    "repay": "kubweza",
    "repayment": "kubweza ngongole",
    "microfinance": "ndalama zazing'ono",
    
    # Mobile Money
    "mobile money": "ndalama zam'manja",
    "mpamba": "mpamba",
    "airtel money": "Airtel Money",
    "tnm mpamba": "TNM Mpamba",
    "PIN": "nambala yachinsinsi",
    "transaction": "ntchito ya ndalama",
    "agent": "wothandizira",
    
    # Fraud & Security
    "fraud": "chinyengo",
    "scam": "chinyengo",
    "security": "chitetezo",
    "protect": "kuteteza",
    "safe": "otetezeka",
    "phishing": "kukokola ndi chinyengo",
    
    # Investment
    "investment": "ndalama zogulitsa",
    "invest": "kugula ndalama",
    "stocks": "magawo a kampani",
    "shares": "magawo",
    "business": "bizinesi",
    "profit": "phindu",
    
    # Insurance
    "insurance": "inshuwaransi",
    "cover": "chitetezo",
    "policy": "dongosolo la inshuwaransi",
    
    # Other Financial Terms
    "tax": "msonkho",
    "receipt": "lipoti",
    "withdraw": "kutulutsa",
    "deposit": "kuika",
    "balance": "ndalama zomwe zatsala"
}

def apply_glossary(text, glossary):
    """
    Apply financial terminology glossary with word boundary protection
    """
    if not text:
        return text
    
    for en_term, ny_term in glossary.items():
        # Use word boundaries to avoid partial replacements
        pattern = r'\b' + re.escape(en_term) + r'\b'
        text = re.sub(pattern, ny_term, text, flags=re.IGNORECASE)
    
    return text

def validate_translation(original_en, translated_ny, back_translated_en):
    """
    Validate translation quality using back-translation
    Returns: (is_valid, quality_score, issues)
    """
    issues = []
    
    # Basic checks
    if not translated_ny or len(translated_ny) < 5:
        issues.append("Translation too short")
        return False, 0.0, issues
    
    # Check for untranslated English (sign of translation failure)
    english_words = ['the', 'and', 'or', 'is', 'where', 'who', 'are', 'how', 'what', 'why', 'when']
    untranslated_count = sum(1 for word in english_words if word in translated_ny.lower())
    if untranslated_count > 2:
        issues.append(f"Contains {untranslated_count} untranslated English words")
    
    # Calculate word overlap between original and back-translation
    original_words = set(original_en.lower().split())
    back_words = set(back_translated_en.lower().split())
    
    overlap = len(original_words & back_words)
    similarity = overlap / max(len(original_words), 1)
    
    # Check for key financial terms preservation
    financial_terms_in_original = [term for term in financial_glossary.keys() 
                                   if term.lower() in original_en.lower()]
    financial_terms_preserved = sum(1 for term in financial_terms_in_original 
                                   if term.lower() in back_translated_en.lower())
    
    preservation_rate = (financial_terms_preserved / max(len(financial_terms_in_original), 1)) if financial_terms_in_original else 0.5
    
    # Combined quality score
    quality_score = 0.6 * similarity + 0.4 * preservation_rate
    
    # Validation threshold
    is_valid = quality_score > 0.3 and untranslated_count <= 2
    
    if quality_score < 0.3:
        issues.append(f"Low quality score: {quality_score:.2f}")
    
    return is_valid, quality_score, issues

def translate_with_validation(text, max_retries=2):
    """
    Translate with quality validation and retry logic
    """
    best_translation = None
    best_score = 0.0
    all_issues = []
    
    for attempt in range(max_retries):
        try:
            # Translate EN → NY
            translated = en_to_ny.translate(text)
            
            if not translated:
                continue
            
            # Apply glossary to improve terminology
            translated = apply_glossary(translated, financial_glossary)
            
            # Back-translate NY → EN for validation
            time.sleep(0.2)  # Rate limiting
            back_translated = ny_to_en.translate(translated)
            
            # Validate
            is_valid, score, issues = validate_translation(text, translated, back_translated)
            
            if issues:
                all_issues.extend(issues)
            
            # Keep best translation
            if score > best_score:
                best_translation = translated
                best_score = score
            
            # If good enough, stop trying
            if is_valid and score > 0.5:
                break
            
            # Small delay before retry
            time.sleep(0.3)
                
        except Exception as e:
            all_issues.append(f"Attempt {attempt + 1} error: {str(e)}")
            time.sleep(0.5)
    
    return best_translation, best_score, all_issues

# TRANSLATE ALL Q&A PAIRS
print(f"\n🔄 Translating {len(en_df)} Q&A pairs with quality control...")

translated_questions = []
translated_answers = []
quality_scores_q = []
quality_scores_a = []
all_issues = []

start_time = time.time()

for idx, row in en_df.iterrows():
    # Translate question
    q_translation, q_score, q_issues = translate_with_validation(row['Question'])
    
    # Translate answer
    time.sleep(0.2)
    a_translation, a_score, a_issues = translate_with_validation(row['Answer'])
    
    translated_questions.append(q_translation if q_translation else row['Question'])
    translated_answers.append(a_translation if a_translation else row['Answer'])
    quality_scores_q.append(q_score)
    quality_scores_a.append(a_score)
    
    # Track issues
    if q_issues or a_issues:
        all_issues.append({
            'index': idx,
            'question': row['Question'][:50],
            'q_issues': q_issues,
            'a_issues': a_issues,
            'q_score': q_score,
            'a_score': a_score
        })
    
    # Progress reporting
    if (idx + 1) % 50 == 0:
        elapsed = time.time() - start_time
        avg_quality = (sum(quality_scores_q[-50:]) + sum(quality_scores_a[-50:])) / 100
        rate = (idx + 1) / elapsed
        remaining = (len(en_df) - idx - 1) / rate
        print(f"✅ Progress: {idx + 1}/{len(en_df)} | "
              f"Avg Quality: {avg_quality:.2f} | "
              f"ETA: {remaining/60:.1f}m")

total_time = time.time() - start_time

# CREATE CHICHEWA DATASET

chichewa_df = pd.DataFrame({
    'question': translated_questions,
    'answer': translated_answers,
    'original_question': en_df['Question'],
    'original_answer': en_df['Answer'],
    'quality_score_q': quality_scores_q,
    'quality_score_a': quality_scores_a,
    'avg_quality': [(q + a) / 2 for q, a in zip(quality_scores_q, quality_scores_a)],
    'language': 'Chichewa',
    'category': en_df.get('Category', 'General')
})

# QUALITY STATISTICS

print(f"\n📊 TRANSLATION QUALITY REPORT")
print("=" * 60)
print(f"⏱️  Total time: {total_time/60:.1f} minutes")
print(f"📈 Questions - Avg: {chichewa_df['quality_score_q'].mean():.2f}, "
      f"Min: {chichewa_df['quality_score_q'].min():.2f}, "
      f"Max: {chichewa_df['quality_score_q'].max():.2f}")
print(f"📈 Answers   - Avg: {chichewa_df['quality_score_a'].mean():.2f}, "
      f"Min: {chichewa_df['quality_score_a'].min():.2f}, "
      f"Max: {chichewa_df['quality_score_a'].max():.2f}")

# Flag low quality entries
low_quality_threshold = 0.25
low_quality = chichewa_df[chichewa_df['avg_quality'] < low_quality_threshold]
print(f"\n⚠️  Low quality translations (< {low_quality_threshold}): "
      f"{len(low_quality)} ({len(low_quality)/len(chichewa_df)*100:.1f}%)")

if len(low_quality) > 0:
    print("\n🔍 Top 5 problematic translations:")
    for idx, row in low_quality.nsmallest(5, 'avg_quality').iterrows():
        print(f"   Row {idx}: Q={row['quality_score_q']:.2f}, A={row['quality_score_a']:.2f}")
        print(f"   Original: {row['original_question'][:60]}...")
        print(f"   Translated: {row['question'][:60]}...")

# Save dataset
chichewa_path = data_folder / "financial_faqs_chichewa.csv"
chichewa_df.to_csv(chichewa_path, index=False, encoding='utf-8-sig')

# Save issues log for manual review
if all_issues:
    issues_df = pd.DataFrame(all_issues)
    issues_path = data_folder / "translation_issues.csv"
    issues_df.to_csv(issues_path, index=False, encoding='utf-8-sig')
    print(f"\n📝 Issues log saved: {issues_path}")

print(f"\n✅ COMPLETE: Saved improved Chichewa dataset")
print(f"📂 Location: {chichewa_path}")
print(f"📊 Total pairs: {len(chichewa_df)}")
print(f"🎯 High quality (>0.5): {len(chichewa_df[chichewa_df['avg_quality'] > 0.5])}")
print(f"⚠️  Needs review (<0.25): {len(low_quality)}")

🚀 Starting IMPROVED Chichewa Translation with Quality Control...
✅ Loaded 500 English FAQs

🔄 Translating 500 Q&A pairs with quality control...
✅ Progress: 50/500 | Avg Quality: 0.67 | ETA: 78.1m
✅ Progress: 100/500 | Avg Quality: 0.61 | ETA: 71.3m
✅ Progress: 150/500 | Avg Quality: 0.58 | ETA: 63.5m
✅ Progress: 200/500 | Avg Quality: 0.61 | ETA: 54.0m
✅ Progress: 250/500 | Avg Quality: 0.59 | ETA: 44.3m
✅ Progress: 300/500 | Avg Quality: 0.64 | ETA: 34.6m
✅ Progress: 350/500 | Avg Quality: 0.59 | ETA: 25.2m
✅ Progress: 400/500 | Avg Quality: 0.64 | ETA: 16.2m
✅ Progress: 450/500 | Avg Quality: 0.27 | ETA: 7.9m
✅ Progress: 500/500 | Avg Quality: 0.42 | ETA: 0.0m

📊 TRANSLATION QUALITY REPORT
⏱️  Total time: 85.5 minutes
📈 Questions - Avg: 0.57, Min: 0.00, Max: 1.00
📈 Answers   - Avg: 0.56, Min: 0.00, Max: 1.00

⚠️  Low quality translations (< 0.25): 36 (7.2%)

🔍 Top 5 problematic translations:
   Row 408: Q=0.00, A=0.00
   Original: What should I know about microcredit for small busine

# re-transalting bad rows

In [ ]:
from transformers import pipeline
from difflib import SequenceMatcher

# Load datasets 
english_path = "../data/financial_faqs_en_500.csv"
chichewa_path = "../data/financial_faqs_chichewa.csv" # Load original for comparison if needed
issues_path = "../data/translation_issues.csv"

english_df = pd.read_csv(english_path)
chichewa_df = pd.read_csv(chichewa_path)
issues_df = pd.read_csv(issues_path)

# Calculate avg_quality for issues_df
if 'q_score' in issues_df.columns and 'a_score' in issues_df.columns:
    issues_df['avg_quality'] = (issues_df['q_score'] + issues_df['a_score']) / 2
else:
    print("Warning: 'q_score' or 'a_score' not found in issues_df. Cannot calculate 'avg_quality'.")
    # Fallback: Use index directly if quality scores are not available in issues_df
    issues_df['avg_quality'] = 0 # Assign a default low value or handle as needed


# Get problematic row indices
# Use the index from the issues_df as it corresponds to the original dataframes
bad_rows = issues_df['index'].tolist() if 'index' in issues_df.columns else issues_df.index.tolist()
print(f"🔍 Found {len(bad_rows)} rows needing retranslation: {bad_rows}")

# Initialize translation pipeline
# Use CPU device for pipeline to avoid potential GPU memory issues if not needed
device = 0 if torch.cuda.is_available() else -1
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-en-ny", src_lang="eng_Latn", tgt_lang="nya_Latn", device=device)


# Create a copy of the original chichewa_df to modify
chichewa_df_revised = chichewa_df.copy()

# Retranslate only problematic rows
print("\n🔄 Retranslating problematic rows...")
for i in bad_rows:
    eng_q = english_df.loc[i, "Question"]
    eng_a = english_df.loc[i, "Answer"]

    try:
        # Ensure inputs are strings and handle potential NaNs
        eng_q_str = str(eng_q) if pd.notna(eng_q) else ""
        eng_a_str = str(eng_a) if pd.notna(eng_a) else ""

        chichewa_q = translator(eng_q_str, max_length=300)[0]['translation_text']
        chichewa_a = translator(eng_a_str, max_length=500)[0]['translation_text']

        chichewa_df_revised.loc[i, "question"] = chichewa_q
        chichewa_df_revised.loc[i, "answer"] = chichewa_a
        print(f"✅ Row {i} fixed")
    except Exception as e:
        print(f"⚠️ Failed row {i}: {e}")
        # Optionally keep original bad translation if retranslation fails
        chichewa_df_revised.loc[i, "question"] = chichewa_df.loc[i, "question"]
        chichewa_df_revised.loc[i, "answer"] = chichewa_df.loc[i, "answer"]

# Save the improved dataset 
output_path = "../data/financial_faqs_chichewa_revised.csv"
chichewa_df_revised.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"\n🎯 Done! Cleaned dataset saved to {output_path}")

# Re-check translation quality and display improved rows
def quick_quality_score(en_text, ny_text):
    """Simple similarity proxy — lower means good translation, high means untranslated"""
    # Convert to string and handle NaN/None values
    en_text = str(en_text) if pd.notna(en_text) else ""
    ny_text = str(ny_text) if pd.notna(ny_text) else ""

    # If either text is empty, return worst score
    if not en_text or not ny_text:
        return 1.0

    # Handle potential division by zero if both texts are the same empty string after lower()
    if en_text.lower() == ny_text.lower() and not en_text:
         return 0.0 # Perfect translation of empty string

    ratio = SequenceMatcher(None, en_text.lower(), ny_text.lower()).ratio()
    return 1 - ratio  # invert: higher = more different (likely better translation)

# Calculate scores for revised dataframe
revised_q_scores = []
revised_a_scores = []

for i in range(len(english_df)):
    en_q = english_df.loc[i, "Question"]
    en_a = english_df.loc[i, "Answer"]
    ny_q = chichewa_df_revised.loc[i, "question"]
    ny_a = chichewa_df_revised.loc[i, "answer"]

    revised_q_scores.append(quick_quality_score(en_q, ny_q))
    revised_a_scores.append(quick_quality_score(en_a, ny_a))

# Add new quality scores to the revised dataframe
chichewa_df_revised['revised_quality_score_q'] = revised_q_scores
chichewa_df_revised['revised_quality_score_a'] = revised_a_scores
chichewa_df_revised['revised_avg_quality'] = (chichewa_df_revised['revised_quality_score_q'] + chichewa_df_revised['revised_quality_score_a']) / 2

# Compute stats
q_avg, q_min, q_max = np.mean(revised_q_scores), np.min(revised_q_scores), np.max(revised_q_scores)
a_avg, a_min, a_max = np.mean(revised_a_scores), np.min(revised_a_scores), np.max(revised_a_scores)

# Count low quality translations based on the *revised* scores
low_quality_threshold = 0.25 # Define threshold here
poor_translations_count = chichewa_df_revised[chichewa_df_revised['revised_avg_quality'] < low_quality_threshold].shape[0]


print("\n📊 TRANSLATION QUALITY RE-CHECK (After Retranslation)")
print(f"📈 Questions - Avg: {q_avg:.2f}, Min: {q_min:.2f}, Max: {q_max:.2f}")
print(f"📈 Answers   - Avg: {a_avg:.2f}, Min: {a_min:.2f}, Max: {a_max:.2f}")
print(f"⚠️  Low quality translations (< {low_quality_threshold} avg score): {poor_translations_count} ({poor_translations_count/len(english_df)*100:.1f}%)")

# Display the retranslated problematic rows and their new scores
print("\n✅ Top 5 previously problematic rows after retranslation:")

# Get original low quality rows based on initial scores from issues_df
# Now that avg_quality is calculated in issues_df
original_low_quality_indices = issues_df[issues_df['avg_quality'] < 0.25]['index'].tolist()

# Display the retranslated versions of these rows from the revised dataframe
if original_low_quality_indices:
    # Limit to top 5 if more exist
    rows_to_display = original_low_quality_indices[:5]

    for idx in rows_to_display:
        if idx < len(chichewa_df_revised):
            row = chichewa_df_revised.loc[idx]
            print(f"\nRow {idx}:")
            print(f"  Original Question: {english_df.loc[idx, 'Question'][:80]}...")
            print(f"  Retranslated Question: {row['question'][:80]}...")
            print(f"  Original Answer: {english_df.loc[idx, 'Answer'][:80]}...")
            print(f"  Retranslated Answer: {row['answer'][:80]}...")
            print(f"  Revised Q Score: {row['revised_quality_score_q']:.2f}, Revised A Score: {row['revised_quality_score_a']:.2f}, Revised Avg Score: {row['revised_avg_quality']:.2f}")
        else:
            print(f"\nWarning: Index {idx} out of bounds in revised dataframe.")

else:
    print("No low quality rows found in the original issues log based on the 0.25 threshold.")


print("\n✅ Re-check complete — dataset quality summary updated.")

🔍 Found 132 rows needing retranslation: [9, 12, 16, 17, 18, 23, 38, 42, 63, 71, 81, 83, 86, 88, 93, 102, 103, 133, 136, 141, 145, 148, 153, 154, 157, 159, 162, 168, 173, 180, 184, 187, 201, 204, 210, 213, 214, 215, 221, 224, 229, 230, 245, 255, 263, 269, 285, 298, 304, 312, 322, 334, 341, 348, 350, 352, 355, 356, 370, 376, 378, 393, 398, 403, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 433, 434, 438, 439, 440, 441, 448, 450, 453, 454, 455, 456, 458, 460, 461, 462, 464, 465, 466, 468, 470, 471, 472, 473, 476, 478, 480, 481, 482, 483, 484, 485, 486, 487, 488, 490, 491, 492, 493, 494, 495, 496, 497]


Device set to use cpu



🔄 Retranslating problematic rows...
✅ Row 9 fixed
✅ Row 12 fixed
✅ Row 16 fixed
✅ Row 17 fixed
✅ Row 18 fixed
✅ Row 23 fixed
✅ Row 38 fixed
✅ Row 42 fixed
✅ Row 63 fixed
✅ Row 71 fixed
✅ Row 81 fixed
✅ Row 83 fixed
✅ Row 86 fixed
✅ Row 88 fixed
✅ Row 93 fixed
✅ Row 102 fixed
✅ Row 103 fixed
✅ Row 133 fixed
✅ Row 136 fixed
✅ Row 141 fixed
✅ Row 145 fixed
✅ Row 148 fixed
✅ Row 153 fixed
✅ Row 154 fixed
✅ Row 157 fixed
✅ Row 159 fixed
✅ Row 162 fixed
✅ Row 168 fixed
✅ Row 173 fixed
✅ Row 180 fixed
✅ Row 184 fixed
✅ Row 187 fixed
✅ Row 201 fixed
✅ Row 204 fixed
✅ Row 210 fixed
✅ Row 213 fixed
✅ Row 214 fixed
✅ Row 215 fixed
✅ Row 221 fixed
✅ Row 224 fixed
✅ Row 229 fixed
✅ Row 230 fixed
✅ Row 245 fixed
✅ Row 255 fixed
✅ Row 263 fixed
✅ Row 269 fixed
✅ Row 285 fixed
✅ Row 298 fixed
✅ Row 304 fixed
✅ Row 312 fixed
✅ Row 322 fixed
✅ Row 334 fixed
✅ Row 341 fixed
✅ Row 348 fixed
✅ Row 350 fixed
✅ Row 352 fixed
✅ Row 355 fixed
✅ Row 356 fixed
✅ Row 370 fixed
✅ Row 376 fixed
✅ Row 378 fixed
✅ R

In [5]:
# Load bilingual data
en_df = pd.read_csv(data_folder / "financial_faqs_en_500.csv")
ch_df = pd.read_csv(data_folder / "financial_faqs_chichewa_revised.csv")

# Debug: Check actual column names
print("English DataFrame columns:", en_df.columns.tolist())
print("Chichewa DataFrame columns:", ch_df.columns.tolist())

# Normalize column names properly
en_df.columns = en_df.columns.str.strip().str.lower()
ch_df.columns = ch_df.columns.str.strip().str.lower()

print("After normalization - English columns:", en_df.columns.tolist())
print("After normalization - Chichewa columns:", ch_df.columns.tolist())

English DataFrame columns: ['Category', 'Question', 'Answer', 'Source']
Chichewa DataFrame columns: ['question', 'answer', 'original_question', 'original_answer', 'quality_score_q', 'quality_score_a', 'avg_quality', 'language', 'category']
After normalization - English columns: ['category', 'question', 'answer', 'source']
After normalization - Chichewa columns: ['question', 'answer', 'original_question', 'original_answer', 'quality_score_q', 'quality_score_a', 'avg_quality', 'language', 'category']


# Load Chichewa Data

In [ ]:
# Add data quality checks
ch_df = pd.read_csv(data_folder / "financial_faqs_chichewa_revised.csv")

# Data validation
print(" Chichewa Dataset Quality Check ")
print(f"Total rows: {len(ch_df)}")
print(f"Null questions: {ch_df['question'].isnull().sum()}")
print(f"Null answers: {ch_df['answer'].isnull().sum()}")
print(f"Empty questions: {(ch_df['question'].str.strip() == '').sum()}")
print(f"Empty answers: {(ch_df['answer'].str.strip() == '').sum()}")

# Remove any problematic rows
initial_count = len(ch_df)
ch_df = ch_df.dropna(subset=['question', 'answer'])
ch_df = ch_df[(ch_df['question'].str.strip() != '') & (ch_df['answer'].str.strip() != '')]
print(f"Cleaned dataset: {len(ch_df)} rows (removed {initial_count - len(ch_df)})")

# Sample check
print("\n Sample Translations")
for i in range(min(3, len(ch_df))):
    print(f"Q: {ch_df.iloc[i]['question'][:80]}...")
    print(f"A: {ch_df.iloc[i]['answer'][:80]}...\\n")

=== Chichewa Dataset Quality Check ===
Total rows: 500
Null questions: 0
Null answers: 0
Empty questions: 0
Empty answers: 0
Cleaned dataset: 500 rows (removed 0)

=== Sample Translations ===
Q: Kodi bajeti ndi chiyani?...
A: Bajeti ndi dongosolo lomwe limawonetsa momwe mungagwiritsire ntchito ndikusunga ...\n
Q: Chifukwa chiyani kuthandizidwa kuli kofunikira?...
A: Zimakuthandizani kuti muchepetse kugwiritsa ntchito ndalama, pewani ngongole, nd...\n
Q: Kodi 50/30/20 ndi chiyani?...
A: Amalimbikitsa kugwiritsa ntchito ndalama 50% pazosowa, 30% za akufuna, ndi 20% p...\n


# Data Merging

In [ ]:
# [ENHANCED VERSION - CSV FAQ + PDF Training Guide with Smart Chunking]

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
print("\n STEP 4: DOCUMENT CHUNKING")

# CHUNKING STRATEGY
# CSV FAQ: No chunking needed - each Q&A pair is already perfectly sized
# PDF Guide: Chunk into smaller pieces for better retrieval

# CSV FAQ Documents - Keep as is (already optimal size)
print("\n Processing CSV FAQ documents ")
csv_chunks = csv_documents  # No chunking needed - Q&A pairs are perfect as-is

print(f"CSV FAQ: {len(csv_chunks)} documents (no chunking needed)")

# PDF Training Guide - Split into chunks
print("\n Processing PDF Training Guide")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,        # Larger chunks for training content
    chunk_overlap=200,       # Good overlap for context
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]  # Try to break at natural boundaries
)

pdf_chunks = text_splitter.split_documents(pdf_documents)

print(f"✅ PDF Guide: {len(pdf_chunks)} chunks created from {len(pdf_documents)} pages")

# COMBINE ALL CHUNKS

all_chunks = csv_chunks + pdf_chunks

print(f"\n🔗 Total chunks ready for embedding: {len(all_chunks)}")
print(f"   - FAQ documents: {len(csv_chunks)}")
print(f"   - PDF chunks: {len(pdf_chunks)}")

# CREATE RAG-OPTIMIZED RECORDS

print("\n CREATING RAG-OPTIMIZED RECORDS ")

merged_records = []

# Process CSV FAQ documents
for idx, doc in enumerate(csv_chunks):
    content = doc.page_content
    metadata = doc.metadata
    
    # Create meaningful title from question
    title = metadata.get('question', content[:100])
    
    merged_records.append({
        'id': f"faq_{idx}",
        'content': content,
        'title': title,
        'language': 'English',
        'lang_code': 'en',
        'category': metadata.get('category', 'General'),
        'source': 'FAQ_CSV',
        'source_detail': metadata.get('source', 'Multiple sources'),
        'page_number': None,  # N/A for CSV
        'chunk_id': idx,
        'content_type': 'faq',
        'document_type': 'FAQ',
        'is_direct_answer': True,
        'rag_ready': True
    })

# Process PDF Training Guide chunks
for idx, chunk in enumerate(pdf_chunks):
    content = chunk.page_content
    metadata = chunk.metadata
    
    # Create title from first line or content
    first_line = content.split('\n')[0].strip()
    title = first_line[:100] if first_line else content[:100]
    title = title + "..." if len(title) == 100 else title
    
    merged_records.append({
        'id': f"pdf_{idx}",
        'content': content,
        'title': title,
        'language': 'English',
        'lang_code': 'en',
        'category': metadata.get('module', 'General'),
        'source': 'TRAINING_GUIDE_PDF',
        'source_detail': 'FINANCIAL_LITERACY.pdf',
        'page_number': metadata.get('page', 0) + 1,  # Convert to 1-based
        'chunk_id': idx,
        'content_type': 'training_guide',
        'document_type': 'Training_Guide',
        'is_direct_answer': False,
        'rag_ready': True
    })

merged_df = pd.DataFrame(merged_records)

print(f"\n✅ RAG-ready dataset: {len(merged_df)} total records")
print(f"\n📊 Source Distribution:")
print(merged_df['source'].value_counts().to_dict())

print(f"\n📂 Category Distribution:")
category_dist = merged_df['category'].value_counts().head(10)
for cat, count in category_dist.items():
    print(f"   - {cat}: {count}")

print(f"\n📄 Content Type Distribution:")
print(merged_df['content_type'].value_counts().to_dict())

if merged_df['page_number'].notna().any():
    pdf_pages = merged_df[merged_df['page_number'].notna()]['page_number']
    print(f"\n📖 PDF Page Range: {int(pdf_pages.min())} to {int(pdf_pages.max())}")

# SAVE RAG-OPTIMIZED CORPUS

print("\n=== 💾 STEP 6: SAVING RAG CORPUS ===")

# Save with comprehensive RAG-specific structure
rag_corpus = {
    'records': merged_df.to_dict(orient="records"),
    'statistics': {
        'total_documents': len(merged_df),
        'faq_documents': len(csv_chunks),
        'pdf_chunks': len(pdf_chunks),
        'pdf_original_pages': len(pdf_documents),
        'sources': {
            'faq_csv': len([r for r in merged_records if r['source'] == 'FAQ_CSV']),
            'training_guide_pdf': len([r for r in merged_records if r['source'] == 'TRAINING_GUIDE_PDF'])
        }
    },
    'rag_metadata': {
        'embedding_model': 'intfloat/multilingual-e5-base',
        'chunking_strategy': {
            'faq': 'no_chunking_qa_pairs',
            'pdf': 'recursive_character_1000_200'
        },
        'source_types': ['faq_csv', 'training_guide_pdf'],
        'document_title': 'Financial Literacy FAQ + Training Guide',
        'version': 'dual-source-enhanced',
        'languages': ['English'],
        'content_types': ['faq', 'training_guide']
    },
    'categories': merged_df['category'].unique().tolist(),
    'metadata_schema': {
        'id': 'Unique identifier (faq_* or pdf_*)',
        'content': 'Main text content for embedding',
        'title': 'Short title/summary',
        'category': 'Topic category',
        'source': 'Source type (FAQ_CSV or TRAINING_GUIDE_PDF)',
        'content_type': 'Content type (faq or training_guide)',
        'is_direct_answer': 'Boolean - true for FAQ, false for training guide',
        'page_number': 'Page number (PDF only)',
        'rag_ready': 'Boolean - ready for RAG system'
    }
}

corpus_path = MODELS_DIR / 'corpus_dual_rag_enhanced.json'
with open(corpus_path, 'w', encoding='utf-8') as f:
    json.dump(rag_corpus, f, ensure_ascii=False, indent=2)

print(f"✅ Saved RAG-optimized corpus: {corpus_path}")
print(f"   Total records: {len(merged_df)}")
print(f"   File size: {corpus_path.stat().st_size / 1024 / 1024:.2f} MB")

# SHOW SAMPLE CHUNKS

print("\n SAMPLE CHUNKS ")

# Show sample FAQ
print("\n1️⃣ Sample FAQ Document:")
faq_sample = merged_df[merged_df['content_type'] == 'faq'].iloc[0]
print(f"   ID: {faq_sample['id']}")
print(f"   Category: {faq_sample['category']}")
print(f"   Title: {faq_sample['title']}")
print(f"   Content: {faq_sample['content'][:200]}...")
print()

# Show sample PDF chunk
print(" Sample PDF Training Guide Chunk:")
pdf_sample = merged_df[merged_df['content_type'] == 'training_guide'].iloc[0]
print(f"   ID: {pdf_sample['id']}")
print(f"   Category: {pdf_sample['category']}")
print(f"   Page: {pdf_sample['page_number']}")
print(f"   Title: {pdf_sample['title']}")
print(f"   Content: {pdf_sample['content'][:200]}...")
print()

# Show another PDF chunk for variety
if len(merged_df[merged_df['content_type'] == 'training_guide']) > 1:
    print(" Another PDF Chunk:")
    pdf_sample2 = merged_df[merged_df['content_type'] == 'training_guide'].iloc[50] if len(merged_df[merged_df['content_type'] == 'training_guide']) > 50 else merged_df[merged_df['content_type'] == 'training_guide'].iloc[-1]
    print(f"   ID: {pdf_sample2['id']}")
    print(f"   Category: {pdf_sample2['category']}")
    print(f"   Page: {pdf_sample2['page_number']}")
    print(f"   Content: {pdf_sample2['content'][:200]}...")

print("\n" + "="*70)
print("✅ Document processing complete! Ready for FAISS index creation.")
print("="*70)


=== 📝 STEP 4: DOCUMENT CHUNKING ===

📋 Processing CSV FAQ documents...
✅ CSV FAQ: 409 documents (no chunking needed)

📚 Processing PDF Training Guide...
✅ PDF Guide: 694 chunks created from 302 pages

🔗 Total chunks ready for embedding: 1103
   - FAQ documents: 409
   - PDF chunks: 694

=== 🎯 STEP 5: CREATING RAG-OPTIMIZED RECORDS ===

✅ RAG-ready dataset: 1103 total records

📊 Source Distribution:
{'TRAINING_GUIDE_PDF': 694, 'FAQ_CSV': 409}

📂 Category Distribution:
   - Budgeting & Saving: 217
   - General: 134
   - Insurance: 126
   - Loan Management: 123
   - Investment: 89
   - Banking & Mobile Money: 82
   - Digital Financial Literacy: 50
   - Consumer Protection & Fraud Awareness: 45
   - Credit & Loans: 45
   - Fraud Prevention & Scams: 39

📄 Content Type Distribution:
{'training_guide': 694, 'faq': 409}

📖 PDF Page Range: 1 to 302

=== 💾 STEP 6: SAVING RAG CORPUS ===
✅ Saved RAG-optimized corpus: ..\models\corpus_dual_rag_enhanced.json
   Total records: 1103
   File size: 1.1

# Embedding Model Setup

In [ ]:
# [ENHANCED VERSION - FAISS vector store with dual-source support]
# RAG-ENHANCED embedding setup with FAISS

embed_model_name = 'intfloat/multilingual-e5-base'
print(f"\n=== 🔮 STEP 7: EMBEDDING & FAISS INDEX CREATION ===")
print(f"Loading embedding model for RAG: {embed_model_name}")
embedder = SentenceTransformer(embed_model_name)

# Load RAG-optimized corpus (UPDATED FILENAME)
corpus_file = MODELS_DIR / 'corpus_dual_rag_enhanced.json'
print(f"\n📂 Loading corpus from: {corpus_file}")

with open(corpus_file, 'r', encoding='utf-8') as f:
    rag_corpus = json.load(f)

corpus_records = rag_corpus['records']
corpus_texts = [record['content'] for record in corpus_records]
corpus_metadata = [rec for rec in corpus_records]

print(f"📊 Loaded {len(corpus_texts)} text chunks")
print(f"   - FAQ documents: {sum(1 for r in corpus_records if r['content_type'] == 'faq')}")
print(f"   - PDF chunks: {sum(1 for r in corpus_records if r['content_type'] == 'training_guide')}")

# Apply E5 "passage: " prefix
corpus_texts_prefixed = ["passage: " + text for text in corpus_texts]

print(f"\n🔮 Encoding {len(corpus_texts)} corpus texts for RAG...")

# Enhanced encoding with batch processing and error handling
try:
    corpus_embeddings = embedder.encode(
        corpus_texts_prefixed,
        batch_size=32,  # Batch processing for better memory management
        convert_to_numpy=True,
        show_progress_bar=True,
        normalize_embeddings=False  # We'll normalize with FAISS for consistency
    )
    print(f"✅ Successfully encoded {len(corpus_embeddings)} embeddings")
except Exception as e:
    print(f"❌ Error during encoding: {e}")
    # Fallback: try without batching
    print("🔄 Trying without batch processing...")
    corpus_embeddings = embedder.encode(
        corpus_texts_prefixed,
        convert_to_numpy=True,
        show_progress_bar=True
    )

print(f"📐 Corpus embeddings shape: {corpus_embeddings.shape}")

# BUILD FAISS VECTOR STORE FOR RAG

print("\n🔨 Building FAISS vector index for RAG...")

dimension = corpus_embeddings.shape[1]

# Choose index type based on dataset size
if len(corpus_embeddings) > 10000:
    print("📈 Large dataset detected, using IndexIVFFlat for better performance...")
    nlist = min(100, len(corpus_embeddings) // 39)  # Calculate nlist parameter
    quantizer = faiss.IndexFlatIP(dimension)
    vector_index = faiss.IndexIVFFlat(quantizer, dimension, nlist)
    
    # Train the index
    print("🏋️ Training FAISS index...")
    vector_index.train(corpus_embeddings)
    vector_index.add(corpus_embeddings)
    vector_index.nprobe = 4  # Balance between speed and accuracy
    index_type = 'IVFFlat'
else:
    # Use simple flat index for smaller datasets
    print("📊 Using IndexFlatIP for optimal accuracy...")
    vector_index = faiss.IndexFlatIP(dimension)  # Inner product for cosine similarity
    # Normalize vectors for cosine similarity
    faiss.normalize_L2(corpus_embeddings)
    vector_index.add(corpus_embeddings)
    index_type = 'FlatIP'

print(f"✅ FAISS index built with {vector_index.ntotal} vectors")
print(f"📐 Embedding dimension: {dimension}")
print(f"🔧 Index type: {index_type}")

# Save FAISS index and metadata
faiss_index_path = MODELS_DIR / 'faiss_dual_index.idx'
try:
    faiss.write_index(vector_index, str(faiss_index_path))
    print(f"💾 FAISS index saved: {faiss_index_path}")
except Exception as e:
    print(f"❌ Error saving FAISS index: {e}")
    raise

# Save enhanced corpus info with FAISS metadata
rag_corpus_info = {
    'records': corpus_records,
    'metadata': corpus_metadata,
    'embeddings_info': {
        'model': embed_model_name,
        'dimension': dimension,
        'normalized': True,
        'index_type': index_type,
        'total_vectors': vector_index.ntotal
    },
    'faiss_index': {
        'total_vectors': vector_index.ntotal,
        'file': 'faiss_dual_index.idx',
        'dataset_size': len(corpus_records)
    },
    'sources': {
        'faq_count': sum(1 for r in corpus_records if r['content_type'] == 'faq'),
        'pdf_count': sum(1 for r in corpus_records if r['content_type'] == 'training_guide')
    },
    'languages': list(set([rec['lang_code'] for rec in corpus_records])),
    'categories': list(set([rec['category'] for rec in corpus_records])),
    'total_entries': len(corpus_records),
    'statistics': rag_corpus.get('statistics', {})
}

corpus_info_path = MODELS_DIR / 'corpus_dual_complete.json'
try:
    with open(corpus_info_path, 'w', encoding='utf-8') as f:
        json.dump(rag_corpus_info, f, ensure_ascii=False, indent=2)
    print(f"💾 Corpus metadata saved: {corpus_info_path}")
except Exception as e:
    print(f"❌ Error saving corpus metadata: {e}")
    raise

print(f"\n✅ Built and saved RAG retrieval index to: {MODELS_DIR.resolve()}")
print(f"📊 Corpus size: {len(corpus_records)} entries")
print(f"   - FAQ documents: {rag_corpus_info['sources']['faq_count']}")
print(f"   - PDF chunks: {rag_corpus_info['sources']['pdf_count']}")
print(f"🌍 Languages: {rag_corpus_info['languages']}")
print(f"📂 Categories: {len(rag_corpus_info['categories'])} unique categories")
print(f"🔍 FAISS index: {vector_index.ntotal} vectors, dimension {dimension}")

# ENHANCED FAISS RETRIEVAL TESTING

print("\n=== 🧪 STEP 8: TESTING FAISS RETRIEVAL ===")

test_queries = {
    "Budgeting": "How to create a budget?",
    "Savings": "What is an emergency fund?",
    "Loans": "What is the difference between secured and unsecured loans?",
    "Mobile Money": "How do I protect my mobile money account?",
    "Insurance": "What is insurance and why do I need it?"
}

# Test retrieval quality with similarity threshold
SIMILARITY_THRESHOLD = 0.3  # Minimum score to consider a match relevant

for category, query in test_queries.items():
    print(f"\n🔍 Query ({category}): '{query}'")
    
    try:
        # Apply E5 query prefix
        query_embedding = embedder.encode(["query: " + query], convert_to_numpy=True)
        
        # Normalize for cosine similarity
        faiss.normalize_L2(query_embedding)
        
        # Search for top 5 results
        scores, indices = vector_index.search(query_embedding, 5)
        
        relevant_results = 0
        for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
            if idx < len(corpus_texts):
                if score > SIMILARITY_THRESHOLD:
                    relevant_results += 1
                    metadata = corpus_metadata[idx]
                    content_type = metadata.get('content_type', 'unknown')
                    source_info = metadata.get('page_number', metadata.get('category', 'N/A'))
                    
                    # Different display for FAQ vs PDF
                    if content_type == 'faq':
                        print(f"  ✅ {i+1}. [FAQ] Score: {score:.3f} | Category: {metadata.get('category', 'N/A')}")
                    else:
                        print(f"  ✅ {i+1}. [PDF] Score: {score:.3f} | Page: {source_info}")
                    
                    print(f"     {corpus_texts[idx][:150]}...")
                else:
                    print(f"  ⚠️ {i+1}. Low score: {score:.3f} (below threshold)")
            else:
                print(f"  ❌ {i+1}. Invalid index: {idx}")
        
        print(f"  📈 Found {relevant_results} relevant results above threshold {SIMILARITY_THRESHOLD}")
        
        if relevant_results == 0:
            print("  💡 Consider lowering the similarity threshold or adding more relevant content")
            
    except Exception as e:
        print(f"  ❌ Error during retrieval test: {e}")

# PERFORMANCE SUMMARY
print(f"\n" + "="*70)
print(f" RAG SYSTEM SUMMARY")
print(f"="*70)
print(f"  Embedding Model: {embed_model_name}")
print(f"  Total Documents: {len(corpus_records)}")
print(f"      - FAQ Documents: {rag_corpus_info['sources']['faq_count']}")
print(f"      - PDF Chunks: {rag_corpus_info['sources']['pdf_count']}")
print(f"    FAISS Index Type: {index_type}")
print(f"    Embedding Dimension: {dimension}")
print(f"    Similarity Threshold: {SIMILARITY_THRESHOLD}")
print(f"    Supported Languages: {rag_corpus_info['languages']}")
print(f"    Categories: {len(rag_corpus_info['categories'])}")
print(f"="*70)

# Save the embedder for future use
embedder_path = MODELS_DIR / 'e5_embedder'
try:
    embedder.save(str(embedder_path))
    print(f"💾 Embedding model saved: {embedder_path}")
except Exception as e:
    print(f"⚠️ Could not save embedder: {e}")

print("\n RAG system is ready for financial literacy chatbot queries!")
print(" All files saved successfully!")
print(f"\n Generated files in {MODELS_DIR}:")
print(f"   1. corpus_dual_rag_enhanced.json")
print(f"   2. faiss_dual_index.idx")
print(f"   3. corpus_dual_complete.json")
print(f"   4. e5_embedder/")


=== 🔮 STEP 7: EMBEDDING & FAISS INDEX CREATION ===
Loading embedding model for RAG: intfloat/multilingual-e5-base

📂 Loading corpus from: ..\models\corpus_dual_rag_enhanced.json
📊 Loaded 1103 text chunks
   - FAQ documents: 409
   - PDF chunks: 694

🔮 Encoding 1103 corpus texts for RAG...


Batches: 100%|██████████| 35/35 [06:25<00:00, 11.01s/it]


✅ Successfully encoded 1103 embeddings
📐 Corpus embeddings shape: (1103, 768)

🔨 Building FAISS vector index for RAG...
📊 Using IndexFlatIP for optimal accuracy...
✅ FAISS index built with 1103 vectors
📐 Embedding dimension: 768
🔧 Index type: FlatIP
💾 FAISS index saved: ..\models\faiss_dual_index.idx
💾 Corpus metadata saved: ..\models\corpus_dual_complete.json

✅ Built and saved RAG retrieval index to: E:\Fin-Chat\models
📊 Corpus size: 1103 entries
   - FAQ documents: 409
   - PDF chunks: 694
🌍 Languages: ['en']
📂 Categories: 16 unique categories
🔍 FAISS index: 1103 vectors, dimension 768

=== 🧪 STEP 8: TESTING FAISS RETRIEVAL ===

🔍 Query (Budgeting): 'How to create a budget?'
  ✅ 1. [PDF] Score: 0.879 | Page: 33.0
     Flip chart paper 
 
STEPS: 
1.  Explain the steps of how to make a budget – 5 minutes 
2.  Make a budget – 25minutes  
 
 
STEP 1: Explain the steps o...
  ✅ 2. [FAQ] Score: 0.879 | Category: Budgeting & Saving
     Category: Budgeting & Saving

Question: How can I sta

In [ ]:
# TEST THE INDEX

print("\n🧪 Step 6: Testing the index...")

test_queries = {
    "English": "How to save money?",
    "Chichewa": "Kodi ndingasunge bwanji ndalama?"
}

for lang, query in test_queries.items():
    print(f"\n🔍 Testing: '{query}' ({lang})")
    
    try:
        # Encode query - FIXED: Remove the "query: " prefix unless your corpus uses it
        query_embedding = embedder.encode([query], convert_to_numpy=True)  # Removed "query: " prefix
        query_embedding = np.array(query_embedding, dtype='float32')  # Ensure correct dtype
        faiss.normalize_L2(query_embedding)
        
        # Search
        scores, indices = vector_index.search(query_embedding, 3)
        
        print(f"Top 3 results:")
        for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
            if idx != -1 and idx < len(corpus_texts):  # Added check for -1 index
                result_text = corpus_texts[idx]
                print(f"  {i+1}. Score: {score:.3f}")
                print(f"     Text: {result_text[:100]}...")  # Better formatting
                if 'metadata' in locals() and idx < len(metadata):
                    print(f"     Source: {metadata[idx].get('source', 'Unknown')}")
            else:
                print(f"  {i+1}. No result (invalid index: {idx})")
                
    except Exception as e:
        print(f"❌ Error testing query '{query}': {e}")


🧪 Step 6: Testing the index...

🔍 Testing: 'How to save money?' (English)
Top 3 results:
  1. Score: 0.862
     Text:  Spend less on parties and festivals 
 Lower expenses on life events such as marriages and funeral...
  2. Score: 0.851
     Text:  Set aside some of your earnings or goods as savings. 
 Learn about the savings services available...
  3. Score: 0.850
     Text:  Make decisions about spending, and saving and investing more in the business 
 
 
It is important ...

🔍 Testing: 'Kodi ndingasunge bwanji ndalama?' (Chichewa)
Top 3 results:
  1. Score: 0.811
     Text:  Set aside some of your earnings or goods as savings. 
 Learn about the savings services available...
  2. Score: 0.810
     Text:  Make decisions about spending, and saving and investing more in the business 
 
 
It is important ...
  3. Score: 0.809
     Text: TRAINER’S GUIDE | PERSONAL FINANCIAL MANAGEMENT:“USE MONEY 
WISELY” 
7 
 
 
PERSONAL FINANCIAL MANAG...


In [ ]:
# STATISTICS

print("\n📊 Final Statistics:")
print(f"  • Total documents: {len(corpus_texts)}")
print(f"  • Embedding dimension: {dimension}")
print(f"  • FAISS vectors: {vector_index.ntotal}")

# FIXED: Check if corpus_metadata exists and has categories
if 'corpus_metadata' in locals() and len(corpus_metadata) > 0:
    # Check if metadata entries have 'category' field
    if 'category' in corpus_metadata[0]:
        unique_categories = set(m['category'] for m in corpus_metadata if 'category' in m)
        print(f"  • Categories: {len(unique_categories)}")
    else:
        print(f"  • Categories: No category information")
else:
    print(f"  • Categories: No metadata available")

print(f"  • Language: English only")
print(f"  • Translation: Will be done in real-time by chatbot")

print("\n✅ COMPLETE! English-only FAISS index ready for real-time translation.")
print("🚀 Now run your chatbot.py with the new RealtimeTranslationChatbot class!")

# OPTIONAL: Show category distribution (only if categories exist)
if 'corpus_metadata' in locals() and len(corpus_metadata) > 0 and 'category' in corpus_metadata[0]:
    print("\n📁 Category Distribution:")
    categories = {}
    for meta in corpus_metadata:
        if 'category' in meta:
            cat = meta['category']
            categories[cat] = categories.get(cat, 0) + 1
    
    if categories:
        for cat, count in sorted(categories.items(), key=lambda x: x[1], reverse=True):
            print(f"  • {cat}: {count} documents")
    else:
        print("  • No categories found in metadata")
else:
    print("\n📁 Category Distribution: No category metadata available")


📊 Final Statistics:
  • Total documents: 694
  • Embedding dimension: 768
  • FAISS vectors: 694
  • Categories: 1
  • Language: English only
  • Translation: Will be done in real-time by chatbot

✅ COMPLETE! English-only FAISS index ready for real-time translation.
🚀 Now run your chatbot.py with the new RealtimeTranslationChatbot class!

📁 Category Distribution:
  • ..\data\FINANCIAL LITERACY.pdf: 694 documents


# distilBert reranker Training

In [ ]:
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torch
import torch.nn as nn # Import nn for custom loss
from pathlib import Path
import pandas as pd
import numpy as np
import re
import time

# Transformers & HF
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# Sentence Transformers (for retrieval)
from sentence_transformers import SentenceTransformer, util
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report, confusion_matrix, accuracy_score
from torch.utils.data import DataLoader, TensorDataset

# Define data_folder and MODELS_DIR
project_root = Path('..')
data_folder = project_root / 'data'
MODELS_DIR = project_root / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ✅ Load tokenizer & classification model (Multilingual DistilBERT)
model_name = 'distilbert-base-multilingual-cased' # Changed model here
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2) # Explicitly set num_labels for binary classification
print(f"✅ Loaded reranker model: {model_name}")

# Load bilingual data & normalize columns
en_df = pd.read_csv(data_folder / "financial_faqs_en_500.csv")
ch_df = pd.read_csv(data_folder / "financial_faqs_chichewa_revised.csv")

en_df.columns = en_df.columns.str.strip().str.lower()
ch_df.columns = ch_df.columns.str.strip().str.lower()
ch_df = ch_df.loc[:, ~ch_df.columns.duplicated()]

# Combine questions and answers
combined_questions = en_df['question'].tolist() + ch_df['question'].tolist()
combined_answers = en_df['answer'].tolist() + ch_df['answer'].tolist()

# Use paraphrase-multilingual-MiniLM-L12-v2 for semantic scoring and force it to CPU
st_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device='cpu') # Force embedding model to CPU
answer_embeddings = st_model.encode(
    ["passage: " + a for a in combined_answers],
    convert_to_tensor=True,
    device='cpu' # Ensure encoding also happens on CPU
)

# Build training pairs with hard negatives
pairs = []
for i, q in enumerate(combined_questions):
    q_prefixed = "query: " + q
    pairs.append({'question': q_prefixed, 'answer': "passage: " + combined_answers[i], 'label': 1})

    # Ensure q_emb encoding is also on CPU
    q_emb = st_model.encode([q_prefixed], convert_to_tensor=True, device='cpu')
    # Move embeddings to CPU before calculating similarity if answer_embeddings is on CPU
    sim_scores = util.cos_sim(q_emb.cpu(), answer_embeddings.cpu())[0] # Ensure similarity is on CPU
    top_neg_indices = sim_scores.topk(5).indices.tolist()
    top_neg_indices = [ni for ni in top_neg_indices if ni != i][:3]

    for ni in top_neg_indices:
        pairs.append({
            'question': q_prefixed,
            'answer': "passage: " + combined_answers[ni],
            'label': 0
        })

# Glossary overlap & reranking
def tokenize(text): return set(re.findall(r'\b\w+\b', str(text).lower()))
financial_terms = set().union(*[tokenize(q) | tokenize(a) for q, a in zip(combined_questions, combined_answers)])

def glossary_overlap(q, a): return len((tokenize(q) | tokenize(a)) & financial_terms)

pairs_df = pd.DataFrame(pairs)
# Ensure embeddings for pairs_df are also on CPU
q_emb_pairs = st_model.encode(pairs_df['question'].tolist(), convert_to_tensor=True, device='cpu')
a_emb_pairs = st_model.encode(pairs_df['answer'].tolist(), convert_to_tensor=True, device='cpu')
pairs_df['embedding_score'] = util.cos_sim(q_emb_pairs, a_emb_pairs).diagonal().cpu().numpy() # Calculate similarity on CPU

pairs_df['glossary_overlap'] = [glossary_overlap(q, a) for q, a in zip(pairs_df['question'], pairs_df['answer'])]

pairs_df['rerank_score'] = (
    0.7 * pairs_df['embedding_score'] +
    0.3 * (pairs_df['glossary_overlap'] / pairs_df['glossary_overlap'].max())
)

# Filter rerank & split
semantic_threshold = 0.3
filtered_df = pairs_df[pairs_df['embedding_score'] >= semantic_threshold].copy()
reranked_df = filtered_df.sort_values(by='rerank_score', ascending=False).reset_index(drop=True)

# Handle class imbalance
print("\n📊 Handling class imbalance with oversampling...")
positive_samples = reranked_df[reranked_df['label'] == 1]
negative_samples = reranked_df[reranked_df['label'] == 0]

num_negative = len(negative_samples)
num_positive = len(positive_samples)
oversample_factor = num_negative // num_positive if num_positive > 0 else 1

if oversample_factor > 1:
    oversampled_positive = pd.concat([positive_samples] * oversample_factor, ignore_index=True)
    balanced_df = pd.concat([negative_samples, oversampled_positive], ignore_index=True)
    print(f"Oversampled positive class by factor {oversample_factor}. Balanced dataset size: {len(balanced_df)}")
else:
    balanced_df = reranked_df.copy()
    print("Oversampling not needed or possible.")

train_df, val_df = train_test_split(balanced_df, test_size=0.1, stratify=balanced_df['label'], random_state=42)

print(f"Train set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Train set label distribution:\n{train_df['label'].value_counts()}")
print(f"Validation set label distribution:\n{val_df['label'].value_counts()}")

# Tokenize datasets for Trainer
from datasets import Dataset
def tokenize_for_trainer(examples):
    return tokenizer(examples['question'], examples['answer'], truncation=True, padding='max_length', max_length=256)

train_dataset_hf = Dataset.from_pandas(train_df)
val_dataset_hf = Dataset.from_pandas(val_df)

# Apply tokenization
train_dataset_hf = train_dataset_hf.map(tokenize_for_trainer, batched=True)
val_dataset_hf = val_dataset_hf.map(tokenize_for_trainer, batched=True)

# Remove original columns after tokenization
train_dataset_hf = train_dataset_hf.remove_columns(['question', 'answer', 'embedding_score', 'glossary_overlap', 'rerank_score', '__index_level_0__'])
val_dataset_hf = val_dataset_hf.remove_columns(['question', 'answer', 'embedding_score', 'glossary_overlap', 'rerank_score', '__index_level_0__'])

# Training arguments for Trainer - FIXED PARAMETER NAMES
training_args = TrainingArguments(
    output_dir=str(MODELS_DIR / 'reranker_trainer_output'),  # Output directory
    num_train_epochs=3,              # Number of training epochs
    per_device_train_batch_size=4,   # Batch size per device during training (reduced)
    per_device_eval_batch_size=8,    # Batch size for evaluation
    warmup_steps=500,                # Number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # Strength of weight decay
    logging_dir=str(MODELS_DIR / 'reranker_logs'),       # Directory for storing logs
    logging_steps=50,
    eval_strategy="epoch",           # FIXED: Changed from evaluation_strategy to eval_strategy
    save_strategy="epoch",           # Save checkpoints every epoch
    load_best_model_at_end=True,     # Load the best model at the end of training
    metric_for_best_model="eval_loss", # Use evaluation loss to find the best model
    greater_is_better=False,         # Lower eval_loss is better
    dataloader_num_workers=2,        # Number of DataLoader workers (adjust based on CPU cores)
)

# Compute class weights for loss function
train_labels_trainer = train_dataset_hf['label']
neg_count_trainer = sum(1 for l in train_labels_trainer if l == 0)
pos_count_trainer = sum(1 for l in train_labels_trainer if l == 1)
total_count_trainer = len(train_labels_trainer)

# Calculate weights inversely proportional to class frequencies
weight_for_0_trainer = total_count_trainer / (2.0 * neg_count_trainer) if neg_count_trainer > 0 else 1.0
weight_for_1_trainer = total_count_trainer / (2.0 * pos_count_trainer) if pos_count_trainer > 0 else 1.0

# Create class weights tensor for BCEWithLogitsLoss (needed manually with Trainer for custom loss)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # Define device for tensor
pos_weight_tensor = torch.tensor(weight_for_1_trainer / weight_for_0_trainer).to(device) if neg_count_trainer > 0 and pos_count_trainer > 0 else None

# Define custom Trainer by overriding training_step
class CustomTrainer(Trainer):
    def training_step(self, model, inputs):
        # Move inputs to device
        for k, v in inputs.items():
            if isinstance(v, torch.Tensor):
                inputs[k] = v.to(self.args.device)

        # Get labels and remove from inputs
        labels = inputs.pop("labels")

        # Forward pass
        outputs = model(**inputs)

        # Calculate loss using BCEWithLogitsLoss
        # Ensure model outputs logits for 2 classes
        if outputs.logits.shape[-1] == 1: # Handle models with single output logit for binary
             logits = outputs.logits.squeeze(-1)
        else: # Assume logits for [negative, positive] or [positive, negative]
             # Check if pos_weight_tensor is used, assume logits[:, 1] is positive class logit
             logits = outputs.logits[:, 1]

        # Ensure labels are float for BCEWithLogitsLoss
        labels = labels.float()

        # Apply pos_weight if available (pos_weight_tensor is defined in the outer scope)
        if pos_weight_tensor is not None:
            loss = F.binary_cross_entropy_with_logits(logits, labels, pos_weight=pos_weight_tensor)
        else:
            loss = F.binary_cross_entropy_with_logits(logits, labels)

        # Handle gradient scaling if using mixed precision (optional but good practice)
        if self.args.n_gpu > 1:
            loss = loss.mean() # mean() to average on multi-gpu.

        if self.args.gradient_accumulation_steps > 1:
            loss = loss / self.args.gradient_accumulation_steps

        # No loss.backward() or optimizer.step() here, Trainer handles that after calling training_step
        return loss

# Initialize the custom trainer with the weighted loss
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_hf,
    eval_dataset=val_dataset_hf,
)

# Train the model
print("\n🏋️ Starting Reranker training with Trainer...")
start_time = time.time()
trainer.train()
end_time = time.time()
print(f"\n✅ Reranker training completed in {(end_time - start_time)/60:.2f} minutes")


# Save the fine-tuned model
rag_reranker_path_trainer = MODELS_DIR / 'distilbert-reranker-rag-trainer'
trainer.save_model(rag_reranker_path_trainer)
tokenizer.save_pretrained(rag_reranker_path_trainer) # Save tokenizer separately as save_model might not save it
print(f"✅ Fine-tuned reranker model saved to: {rag_reranker_path_trainer.resolve()}")

# Evaluation (Optional, Trainer does this during training)
# print("\n📊 Evaluating the trained reranker...")
# eval_results = trainer.evaluate()
# print(eval_results)

print("\n🎉 Reranker training with Trainer complete!")

# Knowledge expansion

In [1]:
# [UPDATED CELL - Enhanced Knowledge Expansion with Universal Financial Corpus]
# Initialize embedding model
print("🔧 Loading embedding model...")
embed_model_name = "intfloat/multilingual-e5-base"
embedder = SentenceTransformer(embed_model_name)
print("✅ Embedding model loaded")

# Define the correct path - adjust this based on your actual structure
MODELS_DIR = Path('./FIN-CHAT/models')  # or '../models' or '../../models' depending on your current location

# Alternative path options - try these if the above doesn't work
possible_paths = [
    Path('./FIN-CHAT/models'),
    Path('../models'),
    Path('../../models'),
    Path('./models'),
    Path('FIN-CHAT/models')
]

# Find the correct path
corpus_path = None
for path in possible_paths:
    test_path = path / 'corpus_rag_complete.json'
    if test_path.exists():
        MODELS_DIR = path
        corpus_path = test_path
        print(f"✅ Found file at: {corpus_path.absolute()}")
        break

if corpus_path is None:
    print("❌ File not found in any expected locations. Let me search more broadly...")
    # Search recursively for the file
    for file_path in Path('.').rglob('corpus_rag_complete.json'):
        if file_path.exists():
            MODELS_DIR = file_path.parent
            corpus_path = file_path
            print(f"✅ Found file at: {corpus_path.absolute()}")
            break

if corpus_path is None:
    print("❌ File still not found. Please check the path and run this cell:")
    print("""
# Manual path setup - run this if the file isn't found
import pathlib
MODELS_DIR = pathlib.Path(r'YOUR_EXACT_PATH_HERE')  # Replace with your actual path
print(f"Looking in: {MODELS_DIR.absolute()}")
print(f"File exists: {(MODELS_DIR / 'corpus_rag_complete.json').exists()}")
    """)
    # Create a minimal corpus as fallback
    MODELS_DIR = Path('./models')
    MODELS_DIR.mkdir(exist_ok=True)
    existing_records = []
    print("📝 Created minimal corpus as fallback")
else:
    # Load the existing corpus
    print(f"📚 Loading existing corpus from: {corpus_path}")
    with open(corpus_path, 'r', encoding='utf-8') as f:
        existing_corpus = json.load(f)
    
    existing_records = existing_corpus['records']
    print(f"✅ Loaded {len(existing_records)} existing documents")

print(f"\n📊 Current corpus status: {len(existing_records)} documents")

# Continue with the knowledge expansion...
print("\n🌐 Expanding knowledge base with Universal Financial Corpus...")

class UniversalFinanceCorpus:
    """Comprehensive universal financial literacy knowledge base"""
    
    def __init__(self):
        self.universal_concepts = self._build_comprehensive_knowledge()
    
    def _build_comprehensive_knowledge(self):
        """Build comprehensive universal financial knowledge"""
        return {
            # ==================== CORE FINANCIAL CONCEPTS ====================
            'financial_literacy': {
                'en': """**Financial Literacy** means having the knowledge and skills to manage money effectively. It includes:
• Budgeting and expense tracking
• Saving and investing
• Understanding credit and debt
• Risk management and insurance
• Financial planning for future goals

Being financially literate helps you make informed decisions and achieve financial stability.""",
                'ny': """**Kudziwa Za Ndalama** kumatanthauza kukhala ndi nzeru ndi luso lokonza ndalama mwanzeru. Zimakhudza:
• Kupanga bajeti ndi kutsatira zogulitsa
• Kusunga ndi kuika ndalama m'katundu
• Kumvetsa ngongole ndi kukakamiza kubweza
• Kusamala ngozi ndi inshuwaransi
• Kupanga mapangidwe a ndalama a zolinga mtsogolo

Kudziwa za ndalama kumakuthandizani kupanga zisankho zoyeretsa ndi kukhala ndi umo wabwino wa ndalama."""
            },
            
            'budgeting': {
                'en': """**Budgeting** is creating a plan for how you'll spend your money each month. Key principles:

• **50/30/20 Rule**: 50% for needs, 30% for wants, 20% for savings/debt
• **Zero-Based Budgeting**: Every dollar has a job - income minus expenses equals zero
• **Track Expenses**: Monitor where your money actually goes
• **Emergency Fund**: Save 3-6 months of essential expenses
• **Review Regularly**: Adjust your budget monthly based on actual spending

Benefits: Control spending, avoid debt, reach financial goals faster.""",
                'ny': """**Kupanga Bajeti** ndi kupanga plan yamene mungagwiritsire ntchito ndalama zanu pa mwezi uliwonse. Mfundo zofunika:

• **Malamulo a 50/30/20**: 50% pazofunika, 30% pazofuna, 20% posunga/kubweza ngongole
• **Bajeti Yopanda Zotsalira**: Ndalamazo zonse zimagwira ntchito - ndalama zilizonse zochokera kupeza ziyenera kugwiritsidwa ntchito
• **Tsatirani Zogulitsa**: Onani komwe ndalama zanu zimayenda
• **Ndalama Za Mwadzidzidzi**: Sungani ndalama za miyezi 3-6 yazofunika
• **Yang'anani Nthaŵi Zonse**: Sinthani bajeti yanu pa mwezi uliwonse kutengera zomwe mwagula

Zabwino: Kulamulira kugula, kupewa ngongole, kufika pazolinga zachuma mwachangu."""
            },

            'saving': {
                'en': """**Saving** means setting aside money for future use instead of spending it now. Types of savings:

• **Emergency Fund**: For unexpected expenses (3-6 months of expenses)
• **Short-Term Savings**: For goals within 1-3 years (vacation, car)
• **Long-Term Savings**: For retirement or major future expenses
• **Sinking Funds**: For predictable irregular expenses (insurance, taxes)

**Saving Strategies**:
• Pay yourself first (automate savings)
• Start small and be consistent
• Use separate accounts for different goals
• Increase savings when income increases""",
                'ny': """**Kusunga Ndalama** kumatanthauza kuika ndalama pambali pogwiritsa ntchito mtsogolo m'malo mogulitsa tsopano. Mitundu yosunga:

• **Ndalama Za Mwadzidzidzi**: Pazogulitsa zosayembekezera (miyezi 3-6 yazogulitsa)
• **Kusunga Kwakanthawi**: Pazolinga mkati mwa chaka 1-3 (ulendo, galimoto)
• **Kusunga Kwakanthawi Yatali**: Pazaka zokhazoka kapena zogulitsa zazikulu mtsogolo
• **Ndalama Zobzala**: Pazogulitsa zosakhala nthawi zonse (inshuwaransi, misonkho)

**Njira Zosunga**:
• Lipirani nokha poyamba (pangani kusunga kukhale kokha)
• Yambani pang'ono ndipo muzikhala osasunthika
• Gwiritsani ntchito maakaunti osiyana pazolinga zosiyana
• Onjezerani kusunga mukapeza ndalama zambiri"""
            },

            'compound_interest': {
                'en': """**Compound Interest** is when you earn interest on both your original money and the interest you've already earned.

**The Power of Compounding**:
• $100 at 5% interest for 1 year = $105
• $105 at 5% interest for 1 year = $110.25
• Your money grows faster over time

**Rule of 72**: Divide 72 by your interest rate to see how long it takes money to double
• 6% interest: 72 ÷ 6 = 12 years to double
• 8% interest: 72 ÷ 8 = 9 years to double

Start investing early to maximize compound interest benefits!""",
                'ny': """**Ndalama Zochuluka** ndi pamene mumapeza ndalama pazomwe mwapeza pa ndalama zanu zoyambira ndi pazomwe mwapeza kale.

**Mphamvu Ya Kuchuluka**:
• $100 pa 5% ndalama zochuluka kwa chaka 1 = $105
• $105 pa 5% ndalama zochuluka kwa chaka 1 = $110.25
• Ndalama zanu zikukula mwachakuya nthawi

**Mlamulo Wa 72**: Gawani 72 ndi mtengo wanu wa ndalama zochuluka kuti muwone kutalika kwa nthawi yomwe ndalama zingawonjezere
• 6% ndalama zochuluka: 72 ÷ 6 = zaka 12 kuwonjezera
• 8% ndalama zochuluka: 72 ÷ 8 = zaka 9 kuwonjezera

Yambani kuika ndalama m'katundu mwachangu kuti mupeze zabwino za ndalama zochuluka!"""
            },

            'credit_score': {
                'en': """**Credit Score** is a number (300-850) that shows your creditworthiness. Factors:

• **Payment History (35%)**: Paying on time
• **Credit Utilization (30%)**: How much credit you're using vs available
• **Credit History Length (15%)**: How long you've had credit
• **Credit Mix (10%)**: Different types of credit (cards, loans)
• **New Credit (10%)**: Recent credit applications

**How to Improve Your Score**:
• Pay all bills on time
• Keep credit card balances below 30% of limit
• Don't close old credit cards
• Limit new credit applications
• Check your credit report regularly for errors""",
                'ny': """**Credit Score** ndi nambala (300-850) yomwe ikuwonetsa kuti mungakwanitse kulandira ngongole. Zomwe zimayesedwa:

• **Mbiri Yolipira (35%)**: Kulipira nthawi yake
• **Kugwiritsa Ntchito Ngongole (30%)**: Kuchuluka kwa ngongole mumagwiritsa ntchito pokhalapo
• **Kutalika Kwa Mbiri Yangongole (15%)**: Kutalika kwa nthawi mwakhala ndi ngongole
• **Mtundu Wa Ngongole (10%)**: Mitundu yosiyana ya ngongole (makadi, malo ogulitsira)
• **Ngongole Yatsopano (10%)**: Mapemphero atsopano a ngongole

**Njira Zowonjezera Score Yanu**:
• Lipirani ma bililo onse nthawi yake
• Sungani ndalama zamakadi ongole pansi pa 30% ya malire
• Musatseke makadi akale ongole
• Chepetsani mapemphero atsopano a ngongole
• Yang'anani lipoti lanu la ngongole nthawi zonse pa zolakwa"""
            },

            'investing_basics': {
                'en': """**Investing** means putting money into assets that can grow over time. Basic principles:

• **Start Early**: Time in market beats timing the market
• **Diversify**: Don't put all eggs in one basket
• **Risk vs Return**: Higher potential returns usually mean higher risk
• **Compound Interest**: Your money earns money on its earnings
• **Dollar-Cost Averaging**: Invest fixed amounts regularly

**Common Investment Types**:
• Stocks (ownership in companies)
• Bonds (loans to governments/companies) 
• Mutual Funds (pooled investments)
• Real Estate (property ownership)
• Savings Accounts (low risk, low return)""",
                'ny': """**Kuika Ndalama M'katundu** kumatanthauza kuika ndalama m'katundu omwe angakule nthawi. Mfundo zoyambira:

• **Yambani Mwachangu**: Nthawi pamisika imapambana kupita pamisika
• **Gawani**: Musaike mazira onse m'basketi imodzi
• **Ngozi Pokwezera**: Kubwezera kwambiri kumatanthauza ngozi yayikulu
• **Ndalama Zochuluka**: Ndalama zanu zimapeza ndalama pazomwe zinapeza
• **Kuyeza Ndalama**: Ikani ndalama zokhazikika nthawi zonse

**Mitundu Yodziwika Ya Investiment**:
• Ma shares (umwini m'makampani)
• Mabondi (ngongole kwa boma/makampani)
• Ndalama zogwirizana (kuika ndalama pamodzi)
• Nyumba (umwini wa katundu)
• Maakaunti osunga (ngozi yochepa, kubweza kochepa)"""
            },

            'emergency_fund': {
                'en': """**Emergency Fund** is money set aside for unexpected expenses. Why it's essential:

• **Job Loss**: Covers expenses while looking for new work
• **Medical Emergencies**: Unexpected health costs
• **Car Repairs**: Vehicle breakdowns
• **Home Repairs**: Urgent household fixes
• **Family Emergencies**: Travel or support needs

**How to Build an Emergency Fund**:
1. Start with a small goal ($500-1000)
2. Save 3-6 months of essential expenses
3. Keep it in a separate, accessible account
4. Only use for true emergencies
5. Replenish if used

**What Counts as Emergency**:
✅ Job loss, medical bills, essential repairs
❌ Vacation, shopping, non-essential purchases""",
                'ny': """**Ndalama Za Mwadzidzidzi** ndi ndalama zoyikidwa pambali pazogulitsa zosayembekezera. Chifukwa chofunika:

• **Kutaya Ntchito**: Imalipira zogulitsa mukafufuza ntchito yatsopano
• **Mwadzidzidzi Wa Mankhwala**: Zogulitsa zosayembekezera za thanzi
• **Kukonza Galimoto**: Kuwonongeka kwa galimoto
• **Kukonza Nyumba**: Zokonza zofunika zapakhomo
• **Mwadzidzidzi Wa Banja**: Ulendo kapena thandizo lofunika

**Njira Yopanga Ndalama Za Mwadzidzidzi**:
1. Yambani ndi lingo lochepa ($500-1000)
2. Sungani ndalama za miyezi 3-6 yazogulitsa zofunika
3. Zigwire ntchito m'akaunti ina yopezeka
4. Gwiritsani ntchito okha pamwadzidzidzi weniweni
5. Bwezerani ngati zagwiritsidwa ntchito

**Zomwe Zimawerengera Ngati Mwadzidzidzi**:
✅ Kutaya ntchito, ma bililo a mankhwala, zokonza zofunika
❌ Ulendo, kugula, kugula zosafunika"""
            }
            # Add more concepts as needed...
        }
    
    def get_all_concepts_as_records(self):
        """Convert universal concepts to corpus records"""
        records = []
        concept_id = 0
        
        for concept_name, concept_data in self.universal_concepts.items():
            # English version
            records.append({
                'id': f"universal_en_{concept_id}",
                'content': f"Q: What is {concept_name.replace('_', ' ')}? A: {concept_data['en']}",
                'question': f"What is {concept_name.replace('_', ' ')}?",
                'answer': concept_data['en'],
                'lang_code': 'en',
                'language': 'English',
                'category': 'Universal Financial Literacy',
                'source': 'universal_knowledge',
                'knowledge_type': 'universal_concept'
            })
            
            # Chichewa version
            records.append({
                'id': f"universal_ny_{concept_id}",
                'content': f"Q: Kodi {concept_name.replace('_', ' ')} ndi chiyani? A: {concept_data['ny']}",
                'question': f"Kodi {concept_name.replace('_', ' ')} ndi chiyani?",
                'answer': concept_data['ny'],
                'lang_code': 'ny',
                'language': 'Chichewa',
                'category': 'Universal Financial Literacy',
                'source': 'universal_knowledge',
                'knowledge_type': 'universal_concept'
            })
            
            concept_id += 1
        
        return records

# Create universal finance corpus
print("📚 Building Universal Financial Corpus...")
universal_corpus = UniversalFinanceCorpus()
universal_records = universal_corpus.get_all_concepts_as_records()
print(f"✅ Created {len(universal_records)} universal concept records")

# Add expert knowledge (your existing expert guidance)
class KnowledgeExpander:
    def __init__(self):
        self.external_sources = {
            'world_bank': 'World Bank Financial Inclusion',
            'cgap': 'CGAP Financial Capability', 
            'undp': 'UNDP Digital Financial Inclusion',
            'gsma': 'GSMA Mobile Money Guidelines'
        }
        
    def fetch_financial_guidelines(self):
        """Fetch basic financial guidelines from trusted sources"""
        print("📚 Creating expert knowledge documents...")
        
        expanded_knowledge = [
            # Basic Financial Principles
            {
                'question': 'What are the basic principles of financial literacy?',
                'answer': 'The five key principles are: 1) Earn - understand your income sources, 2) Save - set aside money for future needs, 3) Protect - insure against risks, 4) Spend - budget wisely, 5) Borrow - manage debt responsibly. These form the foundation of financial wellbeing.',
                'category': 'Financial Literacy Basics',
                'source': 'World Bank Financial Inclusion',
                'language': 'en',
                'knowledge_type': 'expert_guidance'
            },
            {
                'question': 'Kodi mfundo zoyambirira za maphunziro a zachuma ndi zotani?',
                'answer': 'Mfundo zisanu zoyambirira ndi izi: 1) Pezani - muzindikire ndalama zomwe mumapeza, 2) Sungani - sinthani ndalama zambiri zogwiritsidwa ntchito mtsogolo, 3) Tetezani - pezani chitetezo ku ngozi, 4) Gwiritsani ntchito - pangani bajeti mwanzeru, 5) Kangani - lamulirani ngongole mwanzeru. Izi ndi maziko a moyo wabwino wazachuma.',
                'category': 'Financial Literacy Basics',
                'source': 'World Bank Financial Inclusion',
                'language': 'ny',
                'knowledge_type': 'expert_guidance'
            },
            
            # Mobile Money Security
            {
                'question': 'How to keep mobile money secure?',
                'answer': 'Mobile money security tips: 1) Use a strong PIN and never share it, 2) Enable transaction notifications, 3) Only use official apps from app stores, 4) Be cautious of phishing messages, 5) Regularly check your transaction history, 6) Use biometric authentication if available, 7) Keep your phone locked with password.',
                'category': 'Digital Financial Literacy',
                'source': 'GSMA Mobile Money Guidelines',
                'language': 'en',
                'knowledge_type': 'expert_guidance'
            },
            {
                'question': 'Kodi ndisamalire bwanji ndalama zanga zam\'manja?',
                'answer': 'Malangizo oteteza ndalama zam\'manja: 1) Gwiritsani ntchito PIN yovuta ndipo musagawane, 2) Limbikitsani kudziwitsa za ntchito, 3) Gwiritsani ntchito mapulogalamu ovomerezeka okha, 4) Khalani osamala pa mauthenga obwebweta, 5) Yang\'anani mbiri ya ntchito zanu nthawi zonse, 6) Gwiritsani ntchito chizindikiro cha biometric ngati chili, 7) Sungani foni yanu yokiyidwa ndi mawu achinsinsi.',
                'category': 'Digital Financial Literacy',
                'source': 'GSMA Mobile Money Guidelines',
                'language': 'ny',
                'knowledge_type': 'expert_guidance'
            }
        ]
        
        print(f"✅ Added {len(expanded_knowledge)} expert knowledge entries")
        return expanded_knowledge

# Create expander and get expert knowledge
expander = KnowledgeExpander()
expert_knowledge = expander.fetch_financial_guidelines()

# Add expert knowledge to records
expert_records = []
for idx, exp in enumerate(expert_knowledge):
    expert_records.append({
        'id': f"expert_{exp['language']}_{idx}",
        'content': f"Q: {exp['question']} A: {exp['answer']}",
        'question': exp['question'],
        'answer': exp['answer'],
        'lang_code': exp['language'],
        'language': 'English' if exp['language'] == 'en' else 'Chichewa',
        'category': exp['category'],
        'source': exp['source'],
        'knowledge_type': exp['knowledge_type']
    })

# Combine all records: Existing + Universal + Expert
all_records = existing_records + universal_records + expert_records
print(f"📊 Total documents: {len(all_records)} ({len(existing_records)} existing + {len(universal_records)} universal + {len(expert_records)} expert)")

# Build enhanced knowledge graph
print("🔗 Building enhanced knowledge graph...")
knowledge_graph = {
    'concepts': {},
    'categories': {},
    'languages': {},
    'knowledge_types': {}
}

financial_concepts = {
    'savings': ['save', 'saving', 'investment', 'kusunga', 'investiment'],
    'budgeting': ['budget', 'spending', 'expenses', 'bajeti', 'ndalama'],
    'credit': ['loan', 'debt', 'credit', 'borrow', 'ngongole', 'kukanga'],
    'fraud': ['fraud', 'scam', 'security', 'chinyengo', 'chitetezo'],
    'mobile_money': ['mobile money', 'agent', 'PIN', 'mpamba'],
    'investing': ['invest', 'stocks', 'bonds', 'shares', 'investiment', 'ma shares'],
    'insurance': ['insurance', 'cover', 'protection', 'inshuwaransi', 'chitetezo']
}

for record in all_records:
    # Category relationships
    cat = record.get('category', 'General')
    if cat not in knowledge_graph['categories']:
        knowledge_graph['categories'][cat] = []
    knowledge_graph['categories'][cat].append(record['id'])
    
    # Language relationships
    lang = record.get('lang_code', 'unknown')
    if lang not in knowledge_graph['languages']:
        knowledge_graph['languages'][lang] = []
    knowledge_graph['languages'][lang].append(record['id'])
    
    # Knowledge type relationships
    k_type = record.get('knowledge_type', 'general')
    if k_type not in knowledge_graph['knowledge_types']:
        knowledge_graph['knowledge_types'][k_type] = []
    knowledge_graph['knowledge_types'][k_type].append(record['id'])
    
    # Concept relationships
    content_lower = record['content'].lower()
    for concept, keywords in financial_concepts.items():
        if any(kw in content_lower for kw in keywords):
            if concept not in knowledge_graph['concepts']:
                knowledge_graph['concepts'][concept] = []
            knowledge_graph['concepts'][concept].append(record['id'])

print(f"✅ Enhanced knowledge graph built:")
print(f"   • Categories: {len(knowledge_graph['categories'])}")
print(f"   • Concepts: {len(knowledge_graph['concepts'])}")
print(f"   • Languages: {len(knowledge_graph['languages'])}")
print(f"   • Knowledge Types: {len(knowledge_graph['knowledge_types'])}")

# Create embeddings for all documents
print(f"\n🔄 Creating embeddings for {len(all_records)} documents...")
all_texts = [rec['content'] for rec in all_records]
all_embeddings = embedder.encode(all_texts, convert_to_numpy=True, show_progress_bar=True)

# Build enhanced FAISS index
print("🔨 Building enhanced FAISS index...")
dimension = all_embeddings.shape[1]
broad_index = faiss.IndexFlatIP(dimension)
faiss.normalize_L2(all_embeddings)
broad_index.add(all_embeddings)

print(f"✅ Enhanced FAISS index built with {broad_index.ntotal} vectors")

# Save enhanced files
faiss.write_index(broad_index, str(MODELS_DIR / 'broad_knowledge_faiss.idx'))

# Save universal concepts separately for easy access
universal_concepts_path = MODELS_DIR / 'universal_finance_concepts.json'
with open(universal_concepts_path, 'w', encoding='utf-8') as f:
    json.dump(universal_corpus.universal_concepts, f, ensure_ascii=False, indent=2)

broad_knowledge_corpus = {
    'corpus_records': all_records,
    'metadata': all_records,
    'knowledge_graph': knowledge_graph,
    'embeddings_info': {
        'model': embed_model_name,
        'dimension': dimension,
        'normalized': True,
        'index_type': 'FlatIP'
    },
    'faiss_index': {
        'total_vectors': broad_index.ntotal,
        'file': 'broad_knowledge_faiss.idx'
    },
    'statistics': {
        'total_documents': len(all_records),
        'existing_documents': len(existing_records),
        'universal_documents': len(universal_records),
        'expert_documents': len(expert_records),
        'languages': {lang: len(docs) for lang, docs in knowledge_graph['languages'].items()},
        'categories': {cat: len(docs) for cat, docs in knowledge_graph['categories'].items()},
        'knowledge_types': {k_type: len(docs) for k_type, docs in knowledge_graph['knowledge_types'].items()}
    },
    'timestamp': pd.Timestamp.now().isoformat()
}

with open(MODELS_DIR / 'broad_knowledge_rag.json', 'w', encoding='utf-8') as f:
    json.dump(broad_knowledge_corpus, f, ensure_ascii=False, indent=2)

print(f"\n💾 Saved enhanced knowledge system:")
print(f"   • broad_knowledge_faiss.idx: {broad_index.ntotal} vectors")
print(f"   • broad_knowledge_rag.json: {len(all_records)} documents")
print(f"   • universal_finance_concepts.json: {len(universal_corpus.universal_concepts)} concepts")
print(f"   • Existing documents: {len(existing_records)}")
print(f"   • Universal concepts: {len(universal_records)}")
print(f"   • Expert guidance: {len(expert_records)}")

# Display comprehensive statistics
en_docs = len([r for r in all_records if r['lang_code'] == 'en'])
ny_docs = len([r for r in all_records if r['lang_code'] == 'ny'])
other_docs = len(all_records) - en_docs - ny_docs

universal_docs = len([r for r in all_records if r.get('knowledge_type') == 'universal_concept'])
expert_docs = len([r for r in all_records if r.get('knowledge_type') == 'expert_guidance'])
original_docs = len(all_records) - universal_docs - expert_docs

print(f"\n🌍 Comprehensive Distribution:")
print(f"   • English documents: {en_docs}")
print(f"   • Chichewa documents: {ny_docs}")
print(f"   • Other languages: {other_docs}")
print(f"   • Total documents: {len(all_records)}")
print(f"\n📚 Knowledge Type Distribution:")
print(f"   • Universal concepts: {universal_docs}")
print(f"   • Expert guidance: {expert_docs}")
print(f"   • Original documents: {original_docs}")

print(f"\n🎉 Enhanced RAG system with Universal Financial Corpus ready!")
print(f"🚀 Your chatbot can now answer both African-specific AND universal financial questions!")

🔧 Loading embedding model...


NameError: name 'SentenceTransformer' is not defined